# Zynthe NLP Distillation - Kaggle Notebook

Validate BERT → DistilBERT knowledge distillation on T4 GPU.

## Quick Start
1. **Add IMDB Dataset**: Go to Data → Add Data → Search "IMDB" → Add IMDB movie review dataset
2. **Enable GPU**: Click Settings → Accelerator → GPU (T4, 16GB VRAM)
3. **Run All**: Click Runtime → Run All (or Ctrl+F9)

## Runtime
- Estimated time: 5-10 minutes for 1 epoch
- Uses free T4 GPU (30+ hours/week on Kaggle)

## Output
- Student model artifacts saved to `experiments/`
- Training metrics logged to `training_metrics.json`

In [ ]:
# Check GPU availability
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("Warning: No GPU detected. Please enable GPU in Kaggle settings.")

In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate
!pip install -q pyyaml omegaconf
!pip install -q matplotlib seaborn tqdm rich

print("Dependencies installed successfully.")

In [ ]:
# Clone Zynthe repository
# Replace YOUR_USERNAME with your GitHub username
!git clone https://github.com/lakshin7/zynthe.git
%cd zynthe

print("Repository cloned successfully.")

In [ ]:
# Run distillation
!python app/main.py distill --config configs/default.yaml \
  --override train.epochs=1 train.batch_size=8

print("Distillation completed.")

In [ ]:
# Validate outputs
import os
from pathlib import Path

experiments_dir = Path("experiments")

if experiments_dir.exists():
    # Find latest experiment directory
    exp_dirs = [d for d in experiments_dir.iterdir() if d.is_dir()]
    if exp_dirs:
        latest_exp = sorted(exp_dirs, key=lambda x: x.stat().st_mtime)[-1]
        print(f"Latest experiment: {latest_exp.name}")
        
        # Check required artifacts
        required_files = ["config.yaml", "student_model", "training_metrics.json"]
        for req in required_files:
            exists = (latest_exp / req).exists()
            status = "OK" if exists else "MISSING"
            print(f"{req}: {status}")
        
        # Show training metrics summary
        metrics_file = latest_exp / "training_metrics.json"
        if metrics_file.exists():
            import json
            with open(metrics_file) as f:
                metrics = json.load(f)
            print(f"\nTraining Summary:")
            print(f"  Final train loss: {metrics.get('final_train_loss', 'N/A'):.4f}")
            print(f"  Final val loss: {metrics.get('final_val_loss', 'N/A'):.4f}")
            print(f"  Best val loss: {metrics.get('best_val_loss', 'N/A'):.4f}")
    else:
        print("No experiment directories found.")
else:
    print("No experiments directory found. distillation may have failed.")